# ADC Example 6 — Injected Channels

**Target:** STM32F103C8T6 · ADC1 · Regular group + Injected group
**Focus:** High-priority conversions that preempt the regular sequence and store results in **separate registers** (`JDR1–JDR4`).

---
## Concept

Injected channels form a **second conversion group** with higher priority than the regular group. If a regular conversion is in progress when an injected trigger arrives, the ADC pauses the regular sequence, completes the injected group, then resumes the regular group. Injected results go to `JDRx`, **not** `DR`, so they do not interfere with regular DMA transfers[reference:11].

---
## CubeMX Configuration

### Regular Group (background acquisition)
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Enabled |
| `Continuous Conversion Mode` | Enabled |
| `Number Of Conversion` | 3 |
| Rank 1–3 | Channels 0, 1, 4 |

### Injected Group (priority acquisition)
| Parameter | Setting |
| :--- | :--- |
| `Enable Injected Conversions` | **Checked** |
| `Number Of Conversion` (injected) | **2** |
| Injected Rank 1 | Channel 5 |
| Injected Rank 2 | Channel 6 |
| `Injected External Trigger Conversion Source` | Software (`JSWSTART`) or Timer 2 TRGO |

### DMA
- Configure DMA for the **regular group only** (ADC1 → DMA1 Channel 1, Circular, Half Word).
- Injected results are **not** handled by this DMA stream; read them from `JDRx` or in the injected callback.

### NVIC
- Enable **ADC1 and ADC2 global interrupts** (needed for the injected completion callback).

## Generated Initialization (key excerpt)

```c
// Regular group
hadc1.Init.ScanConvMode      = ADC_SCAN_ENABLE;
hadc1.Init.ContinuousConvMode = ENABLE;
hadc1.Init.NbrOfConversion   = 3;

// Injected group (configured via HAL_ADCEx_InjectedConfigChannel)
ADC_InjectionConfTypeDef sConfigInjected = {0};
sConfigInjected.InjectedNbrOfConversion = 2;
sConfigInjected.InjectedSamplingTime    = ADC_SAMPLETIME_55CYCLES_5;
sConfigInjected.InjectedTrigger          = ADC_INJECTED_SOFTWARE_START;
```

## User Code — Start Regular DMA + Injected Interrupt

The regular group runs in the background via DMA. The injected group is triggered separately and its callback reads the `JDR` registers.

In [ ]:
#define N_REG  3
uint16_t reg_buf[N_REG];

// Start regular group with DMA
HAL_ADC_Start_DMA(&hadc1, (uint32_t *)reg_buf, N_REG);

// Start injected group with interrupt
HAL_ADCEx_InjectedStart_IT(&hadc1);

// To trigger an injected conversion manually later:
// HAL_ADCEx_InjectedStart_IT(&hadc1);  // or HAL_ADCEx_InjectedStart()

## Injected Completion Callback

This callback fires when the **entire injected group** (both ranks) has completed[reference:12].

In [ ]:
void HAL_ADCEx_InjectedConvCpltCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        uint32_t inj1 = HAL_ADCEx_InjectedGetValue(hadc, ADC_INJECTED_RANK_1);
        uint32_t inj2 = HAL_ADCEx_InjectedGetValue(hadc, ADC_INJECTED_RANK_2);

        // inj1 = Channel 5 result (JDR1)
        // inj2 = Channel 6 result (JDR2)
        // Use for over-current protection, emergency measurement, etc.
    }
}

## Key Differences from the Regular Group

| | Regular Group | Injected Group |
| :--- | :--- | :--- |
| Priority | Normal | **Preempts regular** |
| Result register | `DR` (shared) | **`JDR1–JDR4`** (separate) |
| DMA support | Yes (standard) | No standard DMA on F103; read `JDRx` |
| Number of ranks | Up to 16 | **Up to 4** |
| Typical use | Background monitoring | Emergency / safety-critical sampling |

## Notes

- Injected conversions are ideal for **motor current sensing**: the regular group samples slow signals, while a PWM-triggered injected group captures the current spike at the exact switching instant.
- The injected callback is `HAL_ADCEx_InjectedConvCpltCallback`, **not** `HAL_ADC_ConvCpltCallback`.
- If you only need one injected channel, set `InjectedNbrOfConversion = 1` and read only `ADC_INJECTED_RANK_1`.